# 11.4 PCA – Exercises

## Exercise 1: Why are Maggie's properties not selling?
No dataset is given, so a property listing dataset is simulated (price, area, bedrooms, age, distance to city centre, crime rate, school rating, days on market). PCA shows which factors drive slow sales.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
n = 300
dist = rng.uniform(1, 40, n); crime = rng.uniform(1, 10, n); school = rng.uniform(3, 10, n)
area = rng.normal(1500, 400, n); bedrooms = np.clip((area/500 + rng.normal(0, .5, n)).round(), 1, 6)
age = rng.uniform(1, 60, n)
price = 200 + 0.15*area - 2*dist - 8*crime + 10*school - 0.8*age + rng.normal(0, 25, n)
days = np.clip(20 + 3*dist + 12*crime - 8*school + 0.02*(price-price.mean())*3 + rng.normal(0, 15, n), 1, None)
props = pd.DataFrame(dict(price=price, area=area, bedrooms=bedrooms, age=age, dist_city=dist,
                          crime=crime, school=school, days_on_market=days))
props.head()

In [ ]:
Xs = StandardScaler().fit_transform(props)
pca = PCA().fit(Xs)
print('Explained variance ratio:', pca.explained_variance_ratio_.round(3))
loadings = pd.DataFrame(pca.components_[:3].T, index=props.columns, columns=['PC1', 'PC2', 'PC3']).round(2)
loadings

In [ ]:
plt.bar(range(1, 9), pca.explained_variance_ratio_); plt.xlabel('Component'); plt.ylabel('Variance ratio'); plt.show()
proj = pca.transform(Xs)
plt.scatter(proj[:, 0], proj[:, 1], c=props.days_on_market, cmap='viridis'); plt.colorbar(label='days on market')
plt.xlabel('PC1'); plt.ylabel('PC2'); plt.show()
print('Days-on-market correlates most with:'); print(props.corr()['days_on_market'].drop('days_on_market').sort_values())

## Exercise 2: Which variable is the principal component?
Loadings table from the PDF.

In [ ]:
import pandas as pd
load = pd.DataFrame({'PC1': [0.190, 0.544, 0.782, 0.365, 0.585, 0.394, 0.985, 0.520, 0.142],
                     'PC2': [0.017, 0.020, -0.605, 0.294, 0.085, -0.273, 0.126, 0.402, 0.150],
                     'PC3': [0.207, 0.204, 0.144, 0.585, 0.234, 0.027, -0.111, 0.519, 0.239]},
                    index=['Climate','Housing','Health','Crime','Transportation','Education','Arts','Recreation','Economy'])
for pc in load:
    strong = load[pc][load[pc].abs() >= 0.5].sort_values(key=abs, ascending=False)
    print(pc, 'is driven by:', dict(strong))
print('Strongest single loading overall:', load.abs().stack().idxmax())

**Answer:** Variables are not themselves principal components; each PC is a weighted combination of them. PC1 (largest variance) is dominated by **Arts (0.985)**, Health (0.782), Transportation (0.585), Housing (0.544) and Recreation (0.520) – a general 'quality of urban life/amenities' component. PC2 contrasts Health (−0.605) with Recreation (0.402). PC3 is driven by Crime (0.585) and Recreation (0.519).